In [1]:
import os
import gradio as gr
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_697/161279274.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Load environment variables from a dotenv file. This allows us to store the OpenAI API key securely instead of typing it directly in the notebook
load_dotenv()

# Retrieve the OpenAI API key from the environment
openai_api_key = os.getenv("OPENAI_API_KEY")

# Check whether the API key was loaded successfully
if openai_api_key:
    print("OpenAI API key loaded successfully.")
else:
    print("OpenAI API key not found. Please check your .env file.")

OpenAI API key loaded successfully.


In [3]:
# Load Nestle's HR policy PDF document using PyPDFLoader
pdf_path = "the_nestle_hr_policy_pdf_2012.pdf"   
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

Number of pages loaded: 8


In [4]:
# Split the loaded document into smaller text chunks for easier processing
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Number of text chunks created: {len(chunks)}")

Number of text chunks created: 20


In [5]:
# The Nestlé HR policy document was successfully loaded using PyPDFLoader and converted into LangChain Document objects. The document was then split into 20 smaller text chunks using RecursiveCharacterTextSplitter. 
# This preprocessing step prepares the document for embedding generation, vector storage, and efficient retrieval of relevant policy information during chatbot interactions.

In [6]:
# Create OpenAI embeddings to convert text chunks into numerical vector representations
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

# Store the text chunk embeddings in a FAISS vector database
vector_db = FAISS.from_documents(documents=chunks, embedding=embeddings)

print("Vector representations created and stored successfully in FAISS.")

Vector representations created and stored successfully in FAISS.


In [7]:
# Create a retriever to search the FAISS vector database
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# Initialize the GPT-3.5 Turbo language model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=openai_api_key)

# Create a prompt template to guide the chatbot's responses
prompt_template = """
You are an AI-powered HR assistant for Nestlé.
Answer employee questions using only the HR policy context provided below.
If the exact answer is not stated, provide the closest relevant information from the context and clearly say that the document does not give a direct answer.
Do not use outside knowledge.
Provide clear, concise, and professional answers.

Context:
{context}

User Question:
{question}

Answer:
"""

# Define a question-answering function
def answer_question(question):
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    prompt = prompt_template.format(context=context, question=question)

    response = llm.invoke(prompt)

    return response.content

In [8]:
# Test the system
question = "What is Nestlé's leave policy?"
answer = answer_question(question)
print(answer)

The document does not provide a direct answer regarding Nestlé's leave policy. However, it emphasizes providing good working conditions, flexible employment possibilities, and a better balance of private and professional life for employees. It also mentions encouraging flexible working conditions whenever possible. For specific details on Nestlé's leave policy, employees should refer to the company's internal HR resources or policies.


In [9]:
#Further Testing
print(answer_question("What are the responsibilities of employees?"))
print(answer_question("What is the company's policy on workplace conduct?"))
print(answer_question("What benefits are available to employees?"))

Employees are responsible for working with their line managers to set challenging objectives, receive regular feedback on their performance, and actively participate in their own professional development. They are also encouraged to express their career objectives and expectations in an open dialogue with their managers. Additionally, employees are expected to align their personal attitudes and professional skills with Nestlé's culture and values.
The company's policy on workplace conduct is based on building relationships of trust and respect with employees at all levels. Managers are committed to creating an environment of mutual trust with their teams, and HR ensures that a respectful dialogue is present and the voice of employees is heard. The company does not tolerate any form of harassment or discrimination. While the document does not provide a specific policy on workplace conduct, it emphasizes the importance of trust, respect, and open communication in the workplace.
The benef

In [10]:
# A retrieval-based question-answering system was successfully developed using FAISS and GPT-3.5 Turbo. When a user submits a question.
# The system first retrieves the most relevant HR policy text chunks through semantic similaritysearch. These retrieved chunks are then provided to GPT-3.5 Turbo as context,enabling the generation of accurate and document-grounded responses.
# The successful test demonstrates that the chatbot can effectively answer HR-related questions using information extracted from the Nestlé HR policy document.

In [11]:
# Build a Gradio chatbot interface for user interaction

def chatbot_interface(user_question):
    answer = answer_question(user_question)
    return answer

interface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question about Nestlé's HR policy...",
        label="Enter your question"
    ),
    outputs=gr.Textbox(
        label="HR Assistant Response"
    ),
    title="AI-Powered Nestlé HR Assistant",
    description="Ask questions about Nestlé's HR policy document and receive answers based on the retrieved policy information."
)

#interface.launch()

IMPORTANT: You are using gradio version 3.45.2, however version 4.44.1 is available, please upgrade.
--------


In [ ]:
# Due to environment restrictions and Gradio dependency conflicts preventing successful interface deployment, the chatbot was validated through direct function testing using answer_question(). 
#The retrieval and question-answering pipeline successfully demonstrated the intended chatbot functionality.